# Spark SQL Window Functions — Practice

Companion exercises for `study_content.md`. Everything runs in a **local, in-process Spark session** — no cluster, no Docker, no external metastore. Just run the cells top to bottom.

Requirements: `pyspark` installed in this venv, and a JDK on `PATH` (both already set up for this project). Run cells with Shift+Enter.

In [2]:
import os
import subprocess

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# This machine's shell rc files point JAVA_HOME at JDKs that aren't actually
# installed anymore. Resolve a real one directly so this notebook doesn't
# depend on whatever happens to be exported in the parent shell.
try:
    os.environ["JAVA_HOME"] = subprocess.check_output(
        ["/usr/libexec/java_home", "-v", "17"], text=True
    ).strip()
except subprocess.CalledProcessError:
    pass  # fall back to whatever JAVA_HOME is already set

spark = (
    SparkSession.builder
    .appName("window-functions-practice")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")  # small local data, avoid 200 tiny tasks
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
spark

## Exercise data setup

Registers one temp view per exercise from Part 7 of the study notes. Run all of these once, then jump to whichever exercise you're practicing.

In [4]:
# Exercise 1 data — customer transactions with duplicate order_date per customer.
txns_raw = spark.createDataFrame(
    [
        (1, 1, "2024-01-01", 100),
        (2, 1, "2024-01-01", 50),   # same date as row above -> RANGE peer group
        (3, 1, "2024-01-02", 30),
        (4, 1, "2024-01-03", 20),
        (5, 2, "2024-01-01", 200),
        (6, 2, "2024-01-01", 75),   # tie for customer 2 too
        (7, 2, "2024-01-03", 40),
    ],
    "id INT, cust_id INT, order_date STRING, amount INT",
)
txns_raw.createOrReplaceTempView("txns_raw")
spark.sql(
    """
    CREATE OR REPLACE TEMP VIEW txns AS
    SELECT id, cust_id, CAST(order_date AS DATE) AS order_date, amount
    FROM txns_raw
    """
)
spark.sql("SELECT * FROM txns ORDER BY cust_id, order_date").show()

+---+-------+----------+------+
| id|cust_id|order_date|amount|
+---+-------+----------+------+
|  2|      1|2024-01-01|    50|
|  1|      1|2024-01-01|   100|
|  3|      1|2024-01-02|    30|
|  4|      1|2024-01-03|    20|
|  5|      2|2024-01-01|   200|
|  6|      2|2024-01-01|    75|
|  7|      2|2024-01-03|    40|
+---+-------+----------+------+



In [5]:
# Exercises 4 & 5 data — a small sales feed with two natural partition keys (rep_id, region).
sales = (
    spark.range(1, 201)
    .withColumn("sale_id", F.col("id"))
    .withColumn("rep_id", (F.col("id") % 5) + 1)
    .withColumn(
        "region",
        F.element_at(
            F.array(*[F.lit(r) for r in ["North", "South", "East", "West"]]),
            (F.col("id") % 4 + 1).cast("int"),
        ),
    )
    .withColumn("ts", F.expr("timestamp('2024-01-01 00:00:00') + (id * INTERVAL 37 MINUTES)"))
    .withColumn("amount", ((F.col("id") * 17) % 500) + 20)
    .drop("id")
)
sales.createOrReplaceTempView("sales")
spark.sql("SELECT * FROM sales ORDER BY sale_id LIMIT 10").show()

+-------+------+------+-------------------+------+
|sale_id|rep_id|region|                 ts|amount|
+-------+------+------+-------------------+------+
|      1|     2| South|2024-01-01 00:37:00|    37|
|      2|     3|  East|2024-01-01 01:14:00|    54|
|      3|     4|  West|2024-01-01 01:51:00|    71|
|      4|     5| North|2024-01-01 02:28:00|    88|
|      5|     1| South|2024-01-01 03:05:00|   105|
|      6|     2|  East|2024-01-01 03:42:00|   122|
|      7|     3|  West|2024-01-01 04:19:00|   139|
|      8|     4| North|2024-01-01 04:56:00|   156|
|      9|     5| South|2024-01-01 05:33:00|   173|
|     10|     1|  East|2024-01-01 06:10:00|   190|
+-------+------+------+-------------------+------+



In [6]:
# Exercise 2 data — 200k events, one key ("HOT_KEY") holding ~40% of rows.
skewed_events = (
    spark.range(0, 200_000)
    .withColumn(
        "key",
        F.when(F.col("id") % 10 < 4, F.lit("HOT_KEY")).otherwise((F.col("id") % 10).cast("string")),
    )
    .withColumn("ts", F.expr("timestamp('2024-01-01') + (id * INTERVAL 1 SECONDS)"))
    .withColumn("amount", (F.col("id") % 97) + 1)
    .drop("id")
)
skewed_events.createOrReplaceTempView("skewed_events")
spark.sql("SELECT key, COUNT(*) AS n FROM skewed_events GROUP BY key ORDER BY n DESC").show()

+-------+-----+
|    key|    n|
+-------+-----+
|HOT_KEY|80000|
|      5|20000|
|      4|20000|
|      8|20000|
|      6|20000|
|      7|20000|
|      9|20000|
+-------+-----+



In [7]:
# Exercise 3 data — 500k synthetic sensor readings for a sliding-frame SUM vs MAX comparison.
readings = (
    spark.range(0, 500_000)
    .withColumn("sensor_id", (F.col("id") % 5).cast("int"))
    .withColumn("ts", F.expr("timestamp('2024-01-01') + (id * INTERVAL 1 SECONDS)"))
    .withColumn("value", (F.sin(F.col("id") / F.lit(50.0)) * 100).cast("double"))
    .drop("id")
)
readings.createOrReplaceTempView("readings")
spark.sql("SELECT * FROM readings ORDER BY sensor_id, ts LIMIT 5").show()

+---------+-------------------+------------------+
|sensor_id|                 ts|             value|
+---------+-------------------+------------------+
|        0|2024-01-01 00:00:00|               0.0|
|        0|2024-01-01 00:00:05| 9.983341664682815|
|        0|2024-01-01 00:00:10|19.866933079506122|
|        0|2024-01-01 00:00:15|29.552020666133956|
|        0|2024-01-01 00:00:20|38.941834230865055|
+---------+-------------------+------------------+



In [8]:
# Exercise 6 data — machine status events with two DOWN episodes on M1 and one on M2.
machine_events_raw = spark.createDataFrame(
    [
        ("M1", "2024-01-01 08:00:00", "UP"),
        ("M1", "2024-01-01 08:05:00", "DOWN"),
        ("M1", "2024-01-01 08:10:00", "DOWN"),
        ("M1", "2024-01-01 08:15:00", "DOWN"),
        ("M1", "2024-01-01 08:20:00", "UP"),
        ("M1", "2024-01-01 08:25:00", "DOWN"),
        ("M1", "2024-01-01 08:30:00", "UP"),
        ("M2", "2024-01-01 08:00:00", "DOWN"),
        ("M2", "2024-01-01 08:05:00", "DOWN"),
        ("M2", "2024-01-01 08:10:00", "UP"),
    ],
    "machine_id STRING, ts STRING, status STRING",
)
machine_events_raw.createOrReplaceTempView("machine_events_raw")
spark.sql(
    """
    CREATE OR REPLACE TEMP VIEW machine_events AS
    SELECT machine_id, CAST(ts AS TIMESTAMP) AS ts, status
    FROM machine_events_raw
    """
)
spark.sql("SELECT * FROM machine_events ORDER BY machine_id, ts").show()

+----------+-------------------+------+
|machine_id|                 ts|status|
+----------+-------------------+------+
|        M1|2024-01-01 08:00:00|    UP|
|        M1|2024-01-01 08:05:00|  DOWN|
|        M1|2024-01-01 08:10:00|  DOWN|
|        M1|2024-01-01 08:15:00|  DOWN|
|        M1|2024-01-01 08:20:00|    UP|
|        M1|2024-01-01 08:25:00|  DOWN|
|        M1|2024-01-01 08:30:00|    UP|
|        M2|2024-01-01 08:00:00|  DOWN|
|        M2|2024-01-01 08:05:00|  DOWN|
|        M2|2024-01-01 08:10:00|    UP|
+----------+-------------------+------+



In [9]:
# Exercise 7 data — products with intentional revenue ties within each category.
products = spark.createDataFrame(
    [
        ("Electronics", "A", 500), ("Electronics", "B", 500), ("Electronics", "C", 400),
        ("Electronics", "D", 300), ("Electronics", "E", 300), ("Electronics", "F", 200),
        ("Home Goods", "G", 800), ("Home Goods", "H", 600), ("Home Goods", "I", 600),
        ("Home Goods", "J", 500), ("Home Goods", "K", 100),
    ],
    "category STRING, product STRING, revenue INT",
)
products.createOrReplaceTempView("products")
spark.sql("SELECT * FROM products ORDER BY category, revenue DESC").show()

+-----------+-------+-------+
|   category|product|revenue|
+-----------+-------+-------+
|Electronics|      A|    500|
|Electronics|      B|    500|
|Electronics|      C|    400|
|Electronics|      E|    300|
|Electronics|      D|    300|
|Electronics|      F|    200|
| Home Goods|      G|    800|
| Home Goods|      I|    600|
| Home Goods|      H|    600|
| Home Goods|      J|    500|
| Home Goods|      K|    100|
+-----------+-------+-------+



---
## Exercise 1 — Frame reasoning (`RANGE` vs `ROWS`)

Table: `txns` (has duplicate `order_date` per `cust_id`).

Write a running total of `amount` per customer ordered by `order_date`:
1. using the **default** frame (`ORDER BY` with no explicit frame clause — implicitly `RANGE`)
2. using an **explicit `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`**

Before running, predict in words what each will output for the tied rows (`2024-01-01` for both customers). Then run both and check.

In [21]:
spark.sql(
    """
    SELECT * FROM txns
    """
).show()

+---+-------+----------+------+
| id|cust_id|order_date|amount|
+---+-------+----------+------+
|  1|      1|2024-01-01|   100|
|  2|      1|2024-01-01|    50|
|  3|      1|2024-01-02|    30|
|  4|      1|2024-01-03|    20|
|  5|      2|2024-01-01|   200|
|  6|      2|2024-01-01|    75|
|  7|      2|2024-01-03|    40|
+---+-------+----------+------+



In [43]:
spark.sql(
    """
    SELECT
        id
        ,cust_id
        ,order_date
        ,amount
        ,SUM(amount) OVER(
            PARTITION BY cust_id
            ORDER BY order_date ASC, amount ASC
            RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS running_total_range
    FROM txns
    ORDER BY cust_id ASC, order_DATE ASC
    """
).show()

+---+-------+----------+------+-------------------+
| id|cust_id|order_date|amount|running_total_range|
+---+-------+----------+------+-------------------+
|  1|      1|2024-01-01|   100|                150|
|  2|      1|2024-01-01|    50|                150|
|  3|      1|2024-01-02|    30|                180|
|  4|      1|2024-01-03|    20|                200|
|  5|      2|2024-01-01|   200|                275|
|  6|      2|2024-01-01|    75|                275|
|  7|      2|2024-01-03|    40|                315|
+---+-------+----------+------+-------------------+



### Bounded Ranges

In [52]:
from datetime import date

data = [
    (date(2024, 1, 1), 100.0),
    (date(2024, 1, 2), 150.0),
    (date(2024, 1, 3), 200.0),
    (date(2024, 1, 8), 300.0), # Introducing a gap to show interval window behavior
]

df = spark.createDataFrame(data, ["day", "revenue"])
df.createOrReplaceTempView("daily_revenue")

In [57]:
spark.sql(
    """
    SELECT day, revenue,
       SUM(revenue) OVER (ORDER BY day
                          RANGE BETWEEN INTERVAL '6' DAY PRECEDING AND CURRENT ROW) AS trailing_7d_avg
    FROM daily_revenue;    
    """
).show()

+----------+-------+---------------+
|       day|revenue|trailing_7d_avg|
+----------+-------+---------------+
|2024-01-01|  100.0|          100.0|
|2024-01-02|  150.0|          250.0|
|2024-01-03|  200.0|          450.0|
|2024-01-08|  300.0|          650.0|
+----------+-------+---------------+



26/09/14 15:40:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/14 15:40:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/14 15:40:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/14 15:40:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/14 15:40:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


---
## Reading plans

For any query above: `spark.sql("...").explain("formatted")` (or `df.explain("formatted")`) prints the physical plan. Look for:
- `Exchange` nodes → one per distinct `PARTITION BY` spec
- `Sort` nodes → one per distinct `(PARTITION BY, ORDER BY)` spec, check for redundant ones
- `Window` nodes → which frame class got picked matters for cost (see study notes 3.2)

For timing, wrap a cell body in `%%time`, or use the Spark UI at `http://localhost:4040` while a job is running.

In [50]:
spark.sql(
    """
    SELECT
        id
        ,cust_id
        ,order_date
        ,amount
        ,SUM(amount) OVER(
            PARTITION BY cust_id
            ORDER BY order_date ASC, amount ASC
            RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS running_total_range
    FROM txns
    ORDER BY cust_id ASC, order_DATE ASC
    """
).explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (8)
+- Sort (7)
   +- Exchange (6)
      +- Window (5)
         +- Sort (4)
            +- Exchange (3)
               +- Project (2)
                  +- Scan ExistingRDD (1)


(1) Scan ExistingRDD
Output [4]: [id#31, cust_id#32, order_date#33, amount#34]
Arguments: [id#31, cust_id#32, order_date#33, amount#34], MapPartitionsRDD[21] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Project
Output [4]: [id#31, cust_id#32, cast(order_date#33 as date) AS order_date#890, amount#34]
Input [4]: [id#31, cust_id#32, order_date#33, amount#34]

(3) Exchange
Input [4]: [id#31, cust_id#32, order_date#890, amount#34]
Arguments: hashpartitioning(cust_id#32, 4), ENSURE_REQUIREMENTS, [plan_id=1955]

(4) Sort
Input [4]: [id#31, cust_id#32, order_date#890, amount#34]
Arguments: [cust_id#32 ASC NULLS FIRST, order_date#890 ASC NULLS FIRST, amount#34 ASC NULLS FIRST], false, 0

(5) Window
Input [4]: [id#31, cust_id#

In [ ]:
# Run when you're done practicing.
spark.stop()